In [3]:
%pip install ijson pymongo underthesea tqdm

  Using cached ijson-3.5.0-cp312-cp312-win_amd64.whl.metadata (24 kB)
  Using cached pymongo-4.17.0-cp312-cp312-win_amd64.whl.metadata (10 kB)
  Using cached underthesea-9.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached dnspython-2.8.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached underthesea_core-3.3.0-cp312-cp312-win_amd64.whl.metadata (5.8 kB)
Using cached ijson-3.5.0-cp312-cp312-win_amd64.whl (55 kB)
Using cached pymongo-4.17.0-cp312-cp312-win_amd64.whl (920 kB)
Using cached dnspython-2.8.0-py3-none-any.whl (331 kB)
Using cached underthesea-9.4.0-py3-none-any.whl (7.3 MB)
Using cached underthesea_core-3.3.0-cp312-cp312-win_amd64.whl (1.2 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)

   ------ --------------------------------- 1/6 [joblib]
   ------ --------------------------------- 1/6 [joblib]
   ------ --------------------------------- 1/6 [joblib]
   ------ --------------------------------- 1/6 [j


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
"""
Telegram Scam Detection — ETL Pipeline
Cấu trúc thực tế của dataset:
  {
    "<channel_id>": {
      "username": str, "title": str, "description": str,
      "creation_date": float (unix), "n_subscribers": int,
      "scam": bool,           ← LABEL đã có sẵn ở cấp channel
      "verified": bool,
      "text_messages": {
        "<msg_id>": {
          "message": str,
          "date": float (unix),
          "author": str | null,
          "is_forwarded": bool,
          "forwarded_from_id": int | null,
          "forwarded_message_date": float | null
        }
      },
      "generic_media": { "<media_id>": {...} }
    }
  }

Chiến thuật: ijson stream theo từng channel → clean từng message → batch insert MongoDB
Cài đặt: pip install ijson pymongo underthesea tqdm

Chạy:
    python telegram_etl_pipeline.py
"""

import os
import re
import json
import logging
import argparse
from pathlib import Path
from datetime import datetime, timezone
from typing import Generator, Optional

import ijson
from pymongo import MongoClient, ASCENDING
from pymongo.errors import BulkWriteError
from tqdm import tqdm

try:
    from underthesea import word_tokenize as vn_tokenize
    VN_TOKENIZER = True
except ImportError:
    VN_TOKENIZER = False
    logging.warning("underthesea chưa cài — bỏ qua Vietnamese tokenization.")

# ─────────────────────────────────────────────────────────────────────────────
# CẤU HÌNH
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR        = Path("D:/Project_DeepLearning/public_db/folder_3")
CHECKPOINT_FILE = Path("checkpoint_v2.json")
MONGO_URI       = "mongodb://localhost:27017"
DB_NAME         = "telegram_scam"
COLLECTION      = "messages"
BATCH_SIZE      = 40_000
MIN_WORD_COUNT  = 5

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(), logging.FileHandler("etl_v2.log")]
)
log = logging.getLogger(__name__)


# ─────────────────────────────────────────────────────────────────────────────
# REGEX — biên dịch 1 lần
# ─────────────────────────────────────────────────────────────────────────────

RE_HTML       = re.compile(r"<[^>]+>")
RE_URL        = re.compile(
    r"(https?://[^\s\]\[\"'<>()]+|"
    r"(?:www\.|t\.me/|bit\.ly/|zalo\.me/)[^\s\]\[\"'<>()]+)",
    re.IGNORECASE
)
RE_EMOJI_ONLY = re.compile(
    r"^[\U0001F000-\U0001FFFF\U00002600-\U000027BF\s]+$"
)
RE_WHITESPACE = re.compile(r"\s+")


# ─────────────────────────────────────────────────────────────────────────────
# CHECKPOINT
# ─────────────────────────────────────────────────────────────────────────────

def load_checkpoint() -> dict:
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"completed_files": [], "total_inserted": 0}

def save_checkpoint(state: dict) -> None:
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)


# ─────────────────────────────────────────────────────────────────────────────
# TẦNG 1 — STREAMING: đọc từng channel qua ijson
# ─────────────────────────────────────────────────────────────────────────────

def stream_channels(json_path: Path) -> Generator[tuple[str, dict], None, None]:
    """
    Dùng ijson stream từng channel (key-value) từ file JSON.
    Cấu trúc: { "<channel_id>": { ...channel_data... }, ... }
    Yield: (channel_id: str, channel_data: dict)
    RAM: O(1 channel) bất kể file lớn.
    """
    with open(json_path, "rb") as f:
        # ijson.kvitems đọc từng cặp key-value ở root level
        for channel_id, channel_data in ijson.kvitems(f, ""):
            yield channel_id, channel_data


# ─────────────────────────────────────────────────────────────────────────────
# TẦNG 2 — CLEANING
# ─────────────────────────────────────────────────────────────────────────────

def extract_urls(text: str) -> list[str]:
    return RE_URL.findall(text)

def clean_text(raw: str, urls: list[str]) -> str:
    text = RE_HTML.sub(" ", raw)
    for url in urls:
        text = text.replace(url, " ")
    text = text.lower().strip()
    return RE_WHITESPACE.sub(" ", text)

def tokenize_vn(text: str) -> str:
    if not VN_TOKENIZER or not text:
        return text
    return vn_tokenize(text, format="text")

def is_valid(text: str) -> bool:
    if not text or not text.strip():
        return False
    if RE_EMOJI_ONLY.match(text):
        return False
    return len(text.split()) >= MIN_WORD_COUNT

def unix_to_dt(ts) -> Optional[datetime]:
    """Chuyển unix timestamp (float/int) hoặc ISO string → datetime UTC."""
    if ts is None:
        return None
    try:
        if isinstance(ts, (int, float)):
            return datetime.fromtimestamp(float(ts), tz=timezone.utc)
        return datetime.fromisoformat(str(ts).replace("Z", "+00:00"))
    except Exception:
        return None


def process_channel_messages(
    channel_id: str,
    channel_data: dict,
    source_file: str
) -> Generator[dict, None, None]:
    """
    Yield từng document MongoDB từ text_messages của 1 channel.
    Label (scam/clean) lấy từ channel-level field `scam` (bool).
    """
    # ── Metadata channel ──────────────────────────────────────────────────────
    label        = bool(channel_data.get("scam", False))
    channel_meta = {
        "channel_id":    channel_id,
        "username":      channel_data.get("username"),
        "title":         channel_data.get("title"),
        "n_subscribers": channel_data.get("n_subscribers"),
        "verified":      channel_data.get("verified", False),
        "creation_date": unix_to_dt(channel_data.get("creation_date")),
    }

    text_messages = channel_data.get("text_messages") or {}

    for msg_id, msg in text_messages.items():
        raw_text = msg.get("message") or ""

        # Bỏ qua message không phải text (None, số, v.v.)
        if not isinstance(raw_text, str):
            raw_text = str(raw_text)

        urls    = extract_urls(raw_text)
        cleaned = clean_text(raw_text, urls)

        if not is_valid(cleaned):
            continue

        tokenized = tokenize_vn(cleaned)

        yield {
            # _id: đảm bảo idempotent khi chạy lại
            "_id": f"{source_file}__{channel_id}__{msg_id}",

            # ── Text features ────────────────────────────────────────────────
            "cleaned_text": tokenized,
            "raw_text":     raw_text,
            "urls_list":    urls,
            "has_url":      len(urls) > 0,
            "word_count":   len(tokenized.split()),

            # ── Label ────────────────────────────────────────────────────────
            # True = scam, False = clean (bool, khớp với field gốc)
            "label":        label,

            # ── Message metadata ─────────────────────────────────────────────
            "msg_id":               msg_id,
            "timestamp":            unix_to_dt(msg.get("date")),
            "author":               msg.get("author"),
            "is_forwarded":         msg.get("is_forwarded", False),
            "forwarded_from_id":    msg.get("forwarded_from_id"),
            "forwarded_msg_date":   unix_to_dt(msg.get("forwarded_message_date")),

            # ── Channel metadata ─────────────────────────────────────────────
            **channel_meta,

            # ── Tracking ─────────────────────────────────────────────────────
            "source_file":  source_file,
            "processed_at": datetime.now(tz=timezone.utc),
        }


# ─────────────────────────────────────────────────────────────────────────────
# MONGO SETUP
# ─────────────────────────────────────────────────────────────────────────────

def setup_mongo(uri: str):
    client = MongoClient(uri, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")
    log.info("✅ MongoDB kết nối thành công: %s", uri)

    col = client[DB_NAME][COLLECTION]
    col.create_index([("label",      ASCENDING)])
    col.create_index([("channel_id", ASCENDING)])
    col.create_index([("timestamp",  ASCENDING)])
    col.create_index([("has_url",    ASCENDING)])
    log.info("✅ Indexes sẵn sàng")
    return client, col


def batch_insert(col, batch: list[dict]) -> int:
    if not batch:
        return 0
    try:
        return len(col.insert_many(batch, ordered=False).inserted_ids)
    except BulkWriteError as e:
        dup = sum(1 for e in e.details.get("writeErrors", []) if e.get("code") == 11000)
        inserted = len(batch) - len(e.details.get("writeErrors", []))
        if dup:
            log.debug("Bỏ qua %d bản ghi trùng (đã insert trước đó)", dup)
        return max(inserted, 0)


# ─────────────────────────────────────────────────────────────────────────────
# PIPELINE ORCHESTRATOR
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(data_dir: Path, batch_size: int, mongo_uri: str) -> None:
    try:
        client, col = setup_mongo(mongo_uri)
    except Exception as e:
        log.error("❌ Kết nối MongoDB thất bại: %s", e)
        return

    state           = load_checkpoint()
    completed       = set(state["completed_files"])
    total_inserted  = state["total_inserted"]
    log.info("📋 Checkpoint: %d file xong, %d bản ghi tổng", len(completed), total_inserted)

    json_files = sorted(data_dir.rglob("*.json"))
    pending    = [f for f in json_files if str(f) not in completed]
    log.info("📁 %d file JSON, còn %d chưa xử lý", len(json_files), len(pending))

    for json_path in pending:
        log.info("▶ %s", json_path.name)
        batch: list[dict] = []
        file_inserted = 0
        file_skipped  = 0

        try:
            with tqdm(desc=json_path.name, unit=" channel", leave=False) as pbar:
                for channel_id, channel_data in stream_channels(json_path):
                    pbar.update(1)

                    for doc in process_channel_messages(
                        channel_id, channel_data, json_path.name
                    ):
                        batch.append(doc)
                        if len(batch) >= batch_size:
                            n = batch_insert(col, batch)
                            file_inserted += n
                            total_inserted += n
                            batch.clear()

                    # Đếm skip: n_messages - những gì đã yield
                    tm = channel_data.get("text_messages") or {}
                    file_skipped += len(tm)  # sẽ điều chỉnh bên dưới

            # Flush batch cuối
            if batch:
                n = batch_insert(col, batch)
                file_inserted += n
                total_inserted += n

            file_skipped = file_skipped - file_inserted  # ước tính

            log.info("✅ %s — inserted: %d | skipped/filtered: ~%d",
                     json_path.name, file_inserted, max(file_skipped, 0))

            completed.add(str(json_path))
            state["completed_files"] = list(completed)
            state["total_inserted"]  = total_inserted
            save_checkpoint(state)

        except KeyboardInterrupt:
            log.warning("⚠ Bị ngắt. Checkpoint đã lưu.")
            break
        except Exception as e:
            log.error("❌ Lỗi xử lý %s: %s", json_path.name, e, exc_info=True)
            continue

    log.info("🏁 Xong. Tổng inserted: %d", total_inserted)
    client.close()


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Telegram ETL Pipeline")
    parser.add_argument("--data-dir",         type=Path, default=DATA_DIR)
    parser.add_argument("--batch-size",       type=int,  default=BATCH_SIZE)
    parser.add_argument("--mongo-uri",        type=str,  default=MONGO_URI)
    parser.add_argument("--reset-checkpoint", action="store_true")
    args, _ = parser.parse_known_args()

    if args.reset_checkpoint and CHECKPOINT_FILE.exists():
        CHECKPOINT_FILE.unlink()
        log.info("🗑  Checkpoint reset.")

    if not args.data_dir.exists():
        log.error("❌ Thư mục không tồn tại: %s", args.data_dir)
        raise SystemExit(1)

    run_pipeline(args.data_dir, args.batch_size, args.mongo_uri)

In [9]:
%pip install pyarrow

   ---------------------------------------- 0.0/27.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/27.4 MB ? eta -:--:--
   - -------------------------------------- 1.3/27.4 MB 4.2 MB/s eta 0:00:07
   --- ------------------------------------ 2.4/27.4 MB 4.8 MB/s eta 0:00:06
   ----- ---------------------------------- 3.7/27.4 MB 5.1 MB/s eta 0:00:05
   ------- -------------------------------- 5.0/27.4 MB 5.2 MB/s eta 0:00:05
   -------- ------------------------------- 6.0/27.4 MB 5.3 MB/s eta 0:00:04
   ---------- ----------------------------- 7.3/27.4 MB 5.5 MB/s eta 0:00:04
   ------------ --------------------------- 8.7/27.4 MB 5.5 MB/s eta 0:00:04
   -------------- ------------------------- 10.0/27.4 MB 5.6 MB/s eta 0:00:04
   ---------------- ----------------------- 11.5/27.4 MB 5.7 MB/s eta 0:00:03
   ------------------ --------------------- 12.8/27.4 MB 5.8 MB/s eta 0:00:03
   --------------


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip install fastparquet

   ---------------------------------------- 0.0/667.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/667.9 kB ? eta -:--:--
   --------------- ------------------------ 262.1/667.9 kB ? eta -:--:--
   ---------------------------------------- 667.9/667.9 kB 2.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------------ --------- 1.3/1.7 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 5.4 MB/s  0:00:00

   -------------------- ------------------- 1/2 [fastparquet]
   -------------------- ------------------- 1/2 [fastparquet]
   -------------------- ------------------- 1/2 [fastparquet]
   ---------------------------------------- 2/2 [fastparquet]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
